# BLIP Fine-Tuning for Text-to-Video Retrieval

This notebook integrates BLIP with the shared MSVD video-text pipeline, fine-tunes it on the official training split, and evaluates text-to-video retrieval on the held-out test split.

The same three equidistant frames, video-level split, top-frame pooling, and rank metrics are used for zero-shot and fine-tuned evaluation. The test split is not used for training or model selection.

In [2]:
!pip install -q transformers accelerate decord pillow tqdm pandas

In [3]:
import os, json, time, random, math
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import BlipProcessor, BlipForImageTextRetrieval
from decord import VideoReader, cpu

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab; using local paths.')

Mounted at /content/drive


In [4]:
# -----------------------------
# Reproducible configuration
# -----------------------------
SEED = 298
DATA_ROOT = '/content/drive/Shared drives/DATA 298A/DATA/MSVD'
MODEL_NAME = 'Salesforce/blip-itm-base-coco'
FRAME_COUNT = 3                 # shared beginning/middle/end configuration
TOP_K_FRAMES = 2                # query-dependent video pooling
IMAGE_SIZE = 224                # source frames; BLIP processor resizes as required
MAX_TEXT_LEN = 64
#TRAIN_BATCH_SIZE = 8
EVAL_TEXT_BATCH_SIZE = 64
#EPOCHS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
NUM_WORKERS = 0                # safest setting for Colab/Decord
EPOCHS = 1
TRAIN_BATCH_SIZE = 4


SMOKE_TEST = True               # set False for the full run
SMOKE_VIDEOS = 8
OUTPUT_DIR = '/content/blip_finetuned_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

Device: cuda


## 1. Shared MSVD loader and standardized frame sampling

The loader accepts the current `msvd_train.json`, `msvd_val.json`, and `msvd_test.json` format. It keeps captions attached to their video IDs and resolves videos from `raw_videos`.

In [5]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f: return json.load(f)

def clean_caption(text):
    return ' '.join(str(text).strip().split())

def make_video_index(video_root):
    paths = {}
    for p in Path(video_root).iterdir():
        if p.is_file():
            paths[p.name] = str(p)
            paths[p.stem] = str(p)
    return paths

def read_split(split, max_videos=None):
    annotation_path = os.path.join(DATA_ROOT, f'msvd_{split}.json')
    video_root = os.path.join(DATA_ROOT, 'raw_videos')
    if not os.path.exists(annotation_path): raise FileNotFoundError(annotation_path)
    if not os.path.isdir(video_root): raise FileNotFoundError(video_root)
    video_index = make_video_index(video_root)
    records = load_json(annotation_path)
    by_video = {}
    for item in records:
        video_name = str(item.get('video', item.get('video_name', '')))
        video_path = video_index.get(video_name) or video_index.get(Path(video_name).stem)
        if not video_path: continue
        video_id = str(item.get('video_id', Path(video_name).stem))
        raw_caps = item.get('caption', item.get('captions', []))
        if isinstance(raw_caps, str): raw_caps = [raw_caps]
        caps = [clean_caption(x) for x in raw_caps if clean_caption(x)]
        if video_id not in by_video:
            by_video[video_id] = {'video_id': video_id, 'video_name': video_name, 'video_path': video_path, 'captions': []}
        by_video[video_id]['captions'].extend(caps)
    videos = list(by_video.values())
    for v in videos:
        v['captions'] = list(dict.fromkeys(v['captions']))
    videos.sort(key=lambda x: x['video_id'])
    if max_videos: videos = videos[:max_videos]
    return videos

def sample_indices(total_frames, frame_count=FRAME_COUNT):
    if total_frames <= 0: raise ValueError('Video has no frames')
    return np.linspace(0, total_frames - 1, min(frame_count, total_frames), dtype=int).tolist()

def read_sampled_frames(video_path, frame_count=FRAME_COUNT):
    vr = VideoReader(video_path, ctx=cpu(0))
    frames = vr.get_batch(sample_indices(len(vr), frame_count)).asnumpy()
    return [Image.fromarray(x).convert('RGB') for x in frames]

In [6]:
split_limit = SMOKE_VIDEOS if SMOKE_TEST else None
train_videos = read_split('train', split_limit)
val_videos = read_split('val', split_limit)
test_videos = read_split('test', split_limit)
print({k: len(v) for k, v in {'train': train_videos, 'val': val_videos, 'test': test_videos}.items()})
assert set(x['video_id'] for x in train_videos).isdisjoint(x['video_id'] for x in val_videos)
assert set(x['video_id'] for x in train_videos).isdisjoint(x['video_id'] for x in test_videos)
assert set(x['video_id'] for x in val_videos).isdisjoint(x['video_id'] for x in test_videos)

{'train': 8, 'val': 8, 'test': 8}


## 2. BLIP-compatible training dataset and smoke test

Training expands each video-caption pair into one example per standardized frame. This provides multiple visual views while preserving the video-level split.

In [7]:
class FrameCaptionDataset(Dataset):
    def __init__(self, videos, processor, max_items=None):
        self.processor = processor
        self.items = []
        for video in videos:
            for caption in video['captions']:
                for frame_index in range(FRAME_COUNT):
                    self.items.append((video, caption, frame_index))
        if max_items: self.items = self.items[:max_items]
    def __len__(self): return len(self.items)
    def __getitem__(self, index):
        video, caption, frame_index = self.items[index]
        vr = VideoReader(video['video_path'], ctx=cpu(0))
        frame = Image.fromarray(vr[sample_indices(len(vr), FRAME_COUNT)[frame_index]].asnumpy()).convert('RGB')
        encoded = self.processor(images=frame, text=caption, padding='max_length', truncation=True, max_length=MAX_TEXT_LEN, return_tensors='pt')
        return {k: v.squeeze(0) for k, v in encoded.items()}

processor = BlipProcessor.from_pretrained(MODEL_NAME)
smoke_dataset = FrameCaptionDataset(train_videos, processor, max_items=min(4, len(train_videos) * FRAME_COUNT))
smoke_loader = DataLoader(smoke_dataset, batch_size=min(2, len(smoke_dataset)), shuffle=False, num_workers=NUM_WORKERS)
smoke_batch = next(iter(smoke_loader))
print({k: tuple(v.shape) for k, v in smoke_batch.items()})
assert 'pixel_values' in smoke_batch and 'input_ids' in smoke_batch and 'attention_mask' in smoke_batch
assert smoke_batch['pixel_values'].ndim == 4
assert smoke_batch['input_ids'].shape[-1] == MAX_TEXT_LEN
print('SMOKE TEST PASSED: BLIP-compatible frame-caption tensors were created.')

preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/456 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

{'input_ids': (2, 64), 'attention_mask': (2, 64), 'pixel_values': (2, 3, 384, 384)}
SMOKE TEST PASSED: BLIP-compatible frame-caption tensors were created.


## 3. Fine-tune BLIP

`return_loss=True` uses BLIP's image-text contrastive objective with in-batch negatives. The model is saved after each epoch and the best validation checkpoint is selected by validation loss.

In [8]:
model = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME).to(DEVICE)
train_dataset = FrameCaptionDataset(train_videos, processor, max_items=(32 if SMOKE_TEST else None))
val_dataset = FrameCaptionDataset(val_videos, processor, max_items=(16 if SMOKE_TEST else None))
train_loader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

def move_batch(batch):
    return {k: v.to(DEVICE) for k, v in batch.items() if k in {'pixel_values', 'input_ids', 'attention_mask'}}

def contrastive_loss(batch):
    image_out = model.vision_model(pixel_values=batch['pixel_values'], return_dict=True)
    image_embeds = F.normalize(model.vision_proj(image_out.last_hidden_state[:, 0, :]), dim=-1)
    text_out = model.text_encoder(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], return_dict=True)
    text_embeds = F.normalize(model.text_proj(text_out.last_hidden_state[:, 0, :]), dim=-1)
    temperature = model.logit_scale.exp().clamp(max=100) if hasattr(model, 'logit_scale') else 1.0
    logits = temperature * image_embeds @ text_embeds.T
    labels = torch.arange(logits.size(0), device=logits.device)
    return 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels))

def run_epoch(loader, training):
    model.train(training); total = 0.0; count = 0
    for batch in tqdm(loader, leave=False, desc='train' if training else 'val'):
        batch = move_batch(batch)
        with torch.set_grad_enabled(training):
            loss = contrastive_loss(batch)
            if training:
                optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
        total += loss.item() * batch['input_ids'].size(0); count += batch['input_ids'].size(0)
    return total / max(count, 1)

history = []
best_val = float('inf')
for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_loader, True)
    val_loss = run_epoch(val_loader, False)
    row = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss}
    history.append(row); print(row)
    epoch_dir = os.path.join(OUTPUT_DIR, f'checkpoint-epoch-{epoch}')
    model.save_pretrained(epoch_dir); processor.save_pretrained(epoch_dir)
    if val_loss < best_val:
        best_val = val_loss
        model.save_pretrained(os.path.join(OUTPUT_DIR, 'best_checkpoint'))
        processor.save_pretrained(os.path.join(OUTPUT_DIR, 'best_checkpoint'))

with open(os.path.join(OUTPUT_DIR, 'training_history.json'), 'w') as f: json.dump(history, f, indent=2)
print('Saved checkpoints and training history to', OUTPUT_DIR)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  895MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/472 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  895MB            

model.safetensors: downloading bytes:           |  0.00B            

...

val:   0%|          | 0/4 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 1.3908287733793259, 'val_loss': 1.386477530002594}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved checkpoints and training history to /content/blip_finetuned_outputs


## 4. Retrieval evaluation

For each query, BLIP scores every sampled frame. The video score is the mean of the strongest two frame similarities, matching the existing zero-shot notebook. Each test video contributes its first cleaned caption as the retrieval query, which keeps the comparison consistent with the baseline.

In [9]:
@torch.inference_mode()
def encode_image_frames(model, processor, images):
    inputs = processor(images=images, return_tensors='pt')
    pixels = inputs['pixel_values'].to(DEVICE)
    out = model.vision_model(pixel_values=pixels, return_dict=True)
    features = model.vision_proj(out.last_hidden_state[:, 0, :])
    return F.normalize(features, dim=-1).cpu()

@torch.inference_mode()
def encode_text_queries(model, processor, texts):
    inputs = processor(text=texts, padding=True, truncation=True, max_length=MAX_TEXT_LEN, return_tensors='pt')
    out = model.text_encoder(input_ids=inputs['input_ids'].to(DEVICE), attention_mask=inputs['attention_mask'].to(DEVICE), return_dict=True)
    features = model.text_proj(out.last_hidden_state[:, 0, :])
    return F.normalize(features, dim=-1).cpu()

@torch.inference_mode()
def evaluate_retrieval(model, processor, videos):
    model.eval(); video_frames = []; video_ids = []; queries = []
    for v in tqdm(videos, desc='Encoding test videos'):
        frames = read_sampled_frames(v['video_path'], FRAME_COUNT)
        video_frames.append(encode_image_frames(model, processor, frames))
        video_ids.append(v['video_id'])
        if v['captions']: queries.append({'video_id': v['video_id'], 'caption': v['captions'][0]})
    text_parts = [encode_text_queries(model, processor, [q['caption'] for q in queries[i:i+EVAL_TEXT_BATCH_SIZE]]) for i in range(0, len(queries), EVAL_TEXT_BATCH_SIZE)]
    text_embeddings = torch.cat(text_parts)
    scores = np.zeros((len(queries), len(videos)), dtype=np.float32)
    for j, frames in enumerate(video_frames):
        frame_scores = text_embeddings @ frames.T
        scores[:, j] = torch.topk(frame_scores, k=min(TOP_K_FRAMES, frames.shape[0]), dim=1).values.mean(dim=1).numpy()
    id_to_index = {x: i for i, x in enumerate(video_ids)}
    ranks = []
    for i, q in enumerate(queries):
        ranking = np.argsort(-scores[i])
        ranks.append(int(np.where(ranking == id_to_index[q['video_id']])[0][0]) + 1)
    ranks = np.asarray(ranks)
    return {'R@1': float(np.mean(ranks <= 1)), 'R@5': float(np.mean(ranks <= 5)), 'R@10': float(np.mean(ranks <= 10)), 'MRR': float(np.mean(1.0 / ranks)), 'MeanRank': float(np.mean(ranks)), 'MedianRank': float(np.median(ranks)), 'num_queries': int(len(ranks)), 'num_videos': int(len(videos))}

fine_tuned_metrics = evaluate_retrieval(model, processor, test_videos)
print(pd.DataFrame([fine_tuned_metrics]))
with open(os.path.join(OUTPUT_DIR, 'fine_tuned_blip_metrics.json'), 'w') as f: json.dump(fine_tuned_metrics, f, indent=2)

Encoding test videos:   0%|          | 0/8 [00:00<?, ?it/s]

    R@1  R@5  R@10       MRR  MeanRank  MedianRank  num_queries  num_videos
0  0.75  1.0   1.0  0.822917     1.625         1.0            8           8


## 5. Compare with the zero-shot BLIP baseline

The baseline values below are the current zero-shot results from `BLIP_eval - Joseph(1).ipynb`. For a fresh run, set `ZERO_SHOT_RESULTS_PATH` to the JSON produced by that notebook so the comparison uses the exact same run.

In [10]:
# Recompute zero-shot BLIP on this notebook's exact frame/evaluation configuration.
# This avoids comparing a 3-frame fine-tuned run with the older 8-frame baseline.
zero_shot_model = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME).to(DEVICE)
zero_shot_metrics = evaluate_retrieval(zero_shot_model, processor, test_videos)
with open(os.path.join(OUTPUT_DIR, 'zero_shot_blip_metrics.json'), 'w') as f: json.dump(zero_shot_metrics, f, indent=2)
del zero_shot_model
if torch.cuda.is_available(): torch.cuda.empty_cache()
comparison_metrics = ['R@1', 'R@5', 'R@10', 'MRR', 'MeanRank', 'MedianRank']
comparison = pd.DataFrame({'Metric': comparison_metrics, 'Zero-shot BLIP': [zero_shot_metrics.get(x) for x in comparison_metrics], 'Fine-tuned BLIP': [fine_tuned_metrics.get(x) for x in comparison_metrics]})
comparison['Change'] = comparison['Fine-tuned BLIP'] - comparison['Zero-shot BLIP']
display(comparison)
with open(os.path.join(OUTPUT_DIR, 'blip_zero_shot_vs_fine_tuned.json'), 'w') as f: json.dump({'zero_shot': zero_shot_metrics, 'fine_tuned': fine_tuned_metrics, 'configuration': {'model': MODEL_NAME, 'frame_count': FRAME_COUNT, 'top_k_frames': TOP_K_FRAMES, 'epochs': EPOCHS, 'learning_rate': LEARNING_RATE, 'dataset_root': DATA_ROOT}}, f, indent=2)

Loading weights:   0%|          | 0/472 [00:00<?, ?it/s]

Encoding test videos:   0%|          | 0/8 [00:00<?, ?it/s]

,Metric,Zero-shot BLIP,Fine-tuned BLIP,Change
0,R@1,0.8750,0.750000,-0.125000
1,R@5,1.0000,1.000000,0.000000
2,R@10,1.0000,1.000000,0.000000
3,MRR,0.9375,0.822917,-0.114583
4,MeanRank,1.1250,1.625000,0.500000
5,MedianRank,1.0000,1.000000,0.000000


## Reproducibility and evidence checklist

- Smoke test output: tensor shapes plus `SMOKE TEST PASSED`.
- Training evidence: `training_history.json`, epoch checkpoints, and `best_checkpoint`.
- Evaluation evidence: `fine_tuned_blip_metrics.json`.
- Comparison evidence: `blip_zero_shot_vs_fine_tuned.json`.
- Configuration: seed 298, BLIP model name, official MSVD split files, three equidistant frames, top-two frame pooling, maximum caption length, optimizer, learning rate, batch size, and epoch count.

For the full experiment, set `SMOKE_TEST = False`, run all cells from the top, and commit this notebook plus the generated JSON/checkpoint evidence to the project branch.